# M7 checker baseline on Colab T4: HHEM-2.1-open (merged, pinned scorer)

**What this runs:** `ml/hhem-baseline/hhem_score.py` on the exact inputs exported on the laptop.
- Pinned HHEM `de9a5e6b`; flan-t5-base `7bcac572`.
- The reviewed code and all files are SHA-256-verified, staged into clean folders and loaded offline.
- Decision: the model's own argmax, P(consistent) > 0.5.
- No labels are needed here, and none should be uploaded.

**Setup:** *Runtime → Change runtime type → T4 GPU*, on a **fresh** runtime (not the gateway runtime). It takes about 5 minutes.

**Upload:** `var/m7-baselines/m7-checker-inputs.jsonl` from the laptop. The notebook checks its SHA-256 (`97bd0387058b…`) and `inputs_sha256` (`7ad97fc97b42…`).

**Return this file:** `hhem-predictions.jsonl`, downloaded by the last cell. Save it on the laptop as `var/m7-baselines/hhem-predictions.jsonl`.

In [ ]:
# @title 1 · Clone the repository and check the GPU
import os, subprocess
REPO = "/content/Enterprise-AI-OS"
REF = "main"  # or the exact commit from the PR
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", REF], check=True)
subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True).stdout)

In [ ]:
# @title 2 · Install the pinned libraries (transformers 4.56.2; torch is Colab's CUDA build)
subprocess.run(f"pip install -q -r {REPO}/ml/hhem-baseline/requirements.txt", shell=True, check=True)

In [ ]:
# @title 3 · Upload the laptop export and verify it
import hashlib, json
from google.colab import files
uploaded = files.upload()  # choose var/m7-baselines/m7-checker-inputs.jsonl
assert list(uploaded) == ["m7-checker-inputs.jsonl"], list(uploaded)
data = uploaded["m7-checker-inputs.jsonl"]
assert hashlib.sha256(data).hexdigest() == "97bd0387058b0d2aab006bee84b11a43fe4d38e20e85c5a32f2f9b205c59c1ec", "not the verified export"
assert json.loads(data.splitlines()[0])["inputs_sha256"] == "7ad97fc97b42ecc213315d452964dcf2bff00d12a5c46cfa3a22b8adcbf48f9b"
open("/content/m7-checker-inputs.jsonl", "wb").write(data)
print("verified export: 37 pairs")

In [ ]:
# @title 4 · Score with the merged HHEM scorer (fresh process; exit 2 = unsupported runtime)
done = subprocess.run(["python", f"{REPO}/ml/hhem-baseline/hhem_score.py", "--inputs", "/content/m7-checker-inputs.jsonl",
                       "--out", "/content/hhem-predictions.jsonl", "--cache-dir", "/content/hhem-pinned"],
                      capture_output=True, text=True)
print(done.stdout, done.stderr[-3000:])
assert done.returncode == 0, done.returncode
print(open("/content/hhem-predictions.jsonl").readline()[:600])

In [ ]:
# @title 5 · Download the predictions
files.download("/content/hhem-predictions.jsonl")